# Gaussian Mixture Model

In [142]:
import pandas as pd
import numpy as np
import plotly.express as px
pd.set_option("display.float_format", lambda x: '%.2f' % x)
np.set_printoptions(suppress=True, precision=10)

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

import optuna

## Ler Dados

In [143]:
df_clientes = pd.read_csv("./dataset/dataset-empresas.csv")
df_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    str    
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    str    
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 23.6 KB


In [144]:
df_clientes.describe()

,faturamento_mensal,numero_de_funcionarios,idade,inovacao
count,500.00,500.00,500.00,500.00
mean,1026715.63,13.69,9.25,4.39
std,420609.46,3.12,2.96,2.90
min,18421.22,2.00,0.00,0.00
25%,763253.58,12.00,7.00,2.00
50%,1022957.08,14.00,9.00,4.00
75%,1295888.52,16.00,11.00,7.00
max,2390677.22,21.00,16.00,9.00


In [145]:
df_clientes.head()

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8


## Preparação dos Dados

In [146]:
X = df_clientes.copy()

numeric_features = ['faturamento_mensal', 'numero_de_funcionarios', 'idade']
categorical_features = ['localizacao', 'atividade_economica']
ordinal_features = ['inovacao']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder()

preprocessor = ColumnTransformer(transformers=[
  ('num', numeric_transformer, numeric_features),
  ('cat', categorical_transformer, categorical_features),
  ('ord', ordinal_transformer, ordinal_features)
])

X_transformed = preprocessor.fit_transform(X=X)

X_transformed.shape

(500, 12)

In [147]:
X_transformed

array([[-0.7463449774, -0.5417919104, -1.1005884861, ...,  0.          ,
         0.          ,  1.          ],
       [-0.5616554761, -1.5035526981,  1.9434485069, ...,  0.          ,
         0.          ,  0.          ],
       [ 0.4058265391,  1.0611427358, -1.7770411512, ...,  0.          ,
         0.          ,  9.          ],
       ...,
       [ 2.8196246022, -1.1829657689,  0.2523168441, ...,  0.          ,
         1.          ,  0.          ],
       [ 1.0332141129, -0.5417919104, -1.4388148187, ...,  0.          ,
         0.          ,  3.          ],
       [-2.0301148644, -0.2212049812, -1.7770411512, ...,  1.          ,
         0.          ,  9.          ]], shape=(500, 12))

## Criação da Função de Tuning com Optuna

In [148]:
def gmm_objective(trial: optuna.Trial):
    n_components = trial.suggest_int('n_components', 3, 10) # número de clusters
    covariance_type = trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag', 'spherical'])

    gmm = GaussianMixture(
      covariance_type=covariance_type, # type: ignore
      n_components=n_components,
      random_state=51,
    )

    gmm.fit(X_transformed)

    bic_gmm = gmm.bic(X_transformed)

    return bic_gmm

## Criar Estudo GMM com Optuna

In [149]:
search_space = {
  'n_components': list(range(3,10+1)),
  'covariance_type': ['full', 'tied', 'diag', 'spherical'],
}
sampler = optuna.samplers.GridSampler(search_space=search_space)
gmm_study = optuna.create_study(
  direction='minimize',
  sampler=sampler,
  study_name="GMM Study",
)

[I 2026-07-29 01:02:30,138] A new study created in memory with name: GMM Study


## Rodar Estudo com Optuna

In [150]:
gmm_study.optimize(
  func=gmm_objective, # type: ignore
  n_trials=len(search_space['n_components']) * len(search_space['covariance_type'])
)

[I 2026-07-29 01:02:30,471] Trial 0 finished with value: -177.4763866661889 and parameters: {'n_components': 6, 'covariance_type': 'tied'}. Best is trial 0 with value: -177.4763866661889.
[I 2026-07-29 01:02:30,655] Trial 1 finished with value: -23479.731809069122 and parameters: {'n_components': 9, 'covariance_type': 'diag'}. Best is trial 1 with value: -23479.731809069122.
[I 2026-07-29 01:02:30,766] Trial 2 finished with value: -239.16103282481197 and parameters: {'n_components': 5, 'covariance_type': 'tied'}. Best is trial 1 with value: -23479.731809069122.
[I 2026-07-29 01:02:32,941] Trial 3 finished with value: -16669.227646443855 and parameters: {'n_components': 5, 'covariance_type': 'full'}. Best is trial 1 with value: -23479.731809069122.
[I 2026-07-29 01:02:33,083] Trial 4 finished with value: 1570.009606922424 and parameters: {'n_components': 3, 'covariance_type': 'diag'}. Best is trial 1 with value: -23479.731809069122.
[I 2026-07-29 01:02:33,398] Trial 5 finished with valu

## Avaliar Melhor Modelo

In [151]:
best_params = gmm_study.best_params
best_params

{'n_components': 7, 'covariance_type': 'full'}

In [152]:
best_gmm = GaussianMixture(
  n_components=best_params['n_components'],
  covariance_type=best_params['covariance_type'],
  random_state=51
)

### Bayesian Information Criteira (BIC)

In [153]:
best_gmm.fit(X_transformed)

best_bic = best_gmm.bic(X_transformed)

In [154]:
print(f"Best bic: {best_bic}")
print(f"Covariance type: {best_params['covariance_type']}")
print(f"Number of Clusters: {best_params['n_components']}")

Best bic: -25850.082549576513
Covariance type: full
Number of Clusters: 7


### Visualizar Clusters

In [155]:
clusters_gmm = best_gmm.predict(X=X_transformed)
clusters_gmm

array([0, 2, 2, 5, 5, 1, 3, 2, 6, 1, 0, 1, 1, 3, 6, 6, 3, 1, 1, 5, 0, 3,
       1, 2, 3, 2, 2, 6, 1, 4, 5, 3, 3, 0, 4, 1, 4, 2, 4, 4, 0, 2, 2, 0,
       0, 4, 2, 5, 1, 1, 6, 1, 5, 4, 2, 0, 6, 4, 3, 4, 1, 3, 1, 3, 6, 2,
       0, 3, 4, 4, 5, 6, 1, 6, 1, 4, 1, 1, 0, 3, 6, 5, 0, 2, 3, 3, 4, 5,
       1, 5, 4, 3, 2, 4, 3, 3, 1, 5, 1, 3, 6, 1, 3, 0, 5, 5, 1, 2, 2, 1,
       6, 4, 5, 2, 4, 0, 6, 1, 1, 6, 5, 5, 5, 6, 4, 4, 5, 6, 1, 2, 6, 2,
       2, 6, 4, 6, 1, 4, 6, 6, 5, 0, 3, 5, 6, 3, 6, 1, 1, 4, 0, 1, 1, 5,
       0, 1, 3, 4, 6, 3, 2, 5, 1, 3, 3, 6, 4, 4, 3, 6, 2, 1, 1, 3, 4, 6,
       0, 3, 4, 4, 1, 5, 5, 6, 5, 0, 6, 6, 3, 1, 3, 3, 3, 4, 1, 3, 0, 3,
       2, 4, 4, 0, 0, 1, 0, 2, 1, 6, 1, 2, 4, 2, 0, 6, 2, 4, 2, 1, 4, 0,
       3, 6, 5, 4, 0, 0, 4, 1, 6, 3, 0, 1, 4, 4, 2, 1, 2, 6, 5, 3, 0, 1,
       0, 6, 0, 4, 1, 4, 3, 5, 5, 5, 1, 6, 0, 4, 4, 5, 2, 2, 3, 6, 5, 3,
       3, 1, 6, 1, 4, 6, 3, 4, 5, 4, 2, 0, 0, 4, 1, 3, 1, 3, 3, 4, 1, 6,
       1, 3, 5, 1, 4, 6, 4, 1, 2, 0, 1, 4, 1, 0, 0,

### Visualizar probabilidades de cada registro estar em um cluster

In [156]:
clusters_gmm_prob = best_gmm.predict_proba(X=X_transformed)
clusters_gmm_prob

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], shape=(500, 7))

In [157]:
df_clientes['cluster'] = clusters_gmm.astype(int)
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao,cluster
0,Comércio,713109.95,12,Rio de Janeiro,6,1,0
1,Comércio,790714.38,9,São Paulo,15,0,2
2,Comércio,1197239.33,17,São Paulo,4,9,2
3,Indústria,449185.78,15,São Paulo,6,0,5
4,Agronegócio,1006373.16,15,São Paulo,15,8,5
5,Serviços,1629562.41,16,Rio de Janeiro,11,4,1
6,Serviços,771179.95,13,Vitória,0,1,3
7,Serviços,707837.61,16,São Paulo,10,6,2
8,Comércio,888983.66,17,Belo Horizonte,10,1,6
9,Indústria,1098512.64,13,Rio de Janeiro,9,3,1


## Resultados

### Cluster: Faturamento x Idade
Não há um impacto do cluster de forma clara nessa relação.

In [158]:
px.scatter(
  data_frame=df_clientes,
  x='idade',
  y='faturamento_mensal',
  color='cluster'
)

### Cluster: Inovação x Faturamento

In [159]:
px.scatter(
  data_frame=df_clientes,
  x='inovacao',
  y='faturamento_mensal',
  color='cluster'
)

### Cluster: Inovação x Idade

In [160]:
px.scatter(
  data_frame=df_clientes,
  x='inovacao',
  y='idade',
  color='cluster'
)